In [1]:
import os
print(os.environ.get('LD_LIBRARY_PATH'))
print(os.environ.get('LD_PRELOAD'))

/tools/Xilinx/Vitis_HLS/2022.2/lib/lnx64.o:/tools/Xilinx/Vitis_HLS/2022.2/lnx64/lib/csim:/tools/Xilinx/Vitis_HLS/2022.2/tps/lnx64/gcc-8.3.0/lib64:
None


In [2]:
import os
os.environ["LD_PRELOAD"] = "/usr/lib/x86_64-linux-gnu/libstdc++.so.6"
print(os.environ.get('LD_PRELOAD'))

/usr/lib/x86_64-linux-gnu/libstdc++.so.6


In [ ]:
import os
os.environ['PATH'] = '/home/fitba/tools/Xilinx/Vitis_HLS/2022.2/bin:' + os.environ['PATH'] 
import tensorflow as tf
import hls4ml
import numpy as np

# =====================================================================
# 1. BLOQUE DE PARAMETRIZACIÓN GLOBAL (Link con Etapa 4)
# =====================================================================
# A. Parámetros de Entrenamiento (Origen)
METODO_COMPRESION = "Destillation"
EPOCHS_TRAIN      = 15
BATCH_SIZE_TRAIN  = 128
NOMBRE_MODELO     = "modelo_estudiante_destilado_qat"

# B. Rutas Dinámicas
DIR_MODELOS   = f"/mnt/Datos/Julian_Alvarez-TFG/IDS-IOT/models/{METODO_COMPRESION}_{EPOCHS_TRAIN}epochs_{BATCH_SIZE_TRAIN}batch"
RUTA_MODELO   = f"{DIR_MODELOS}/{NOMBRE_MODELO}.h5"
DIR_HLS_OUT   = f"../../vitis/hls4ml/pynq_{NOMBRE_MODELO}_prj"

# C. Parámetros de Arquitectura de Hardware (PYNQ-Z2)
TARGET_BOARD  = 'pynq-z2'
TARGET_PART   = 'xc7z020clg400-1'
IO_PROTOCOL   = 'io_stream'   # Requisito para AXI-DMA
PERIODO_RELOJ = 10.0          # ns. 10 = 100 MHz = FCLK_CLK0 del overlay PYNQ.
                              # OJO: el defecto de hls4ml es 5 ns (200 MHz), que
                              # el xc7z020-1 (grado de velocidad más lento de la
                              # familia) no sostiene con este diseño.

# =====================================================================
# D. PRECISIÓN NUMÉRICA  --  la parte que decide si el IP funciona
# =====================================================================
# Medido sobre 40.000 paquetes reservados, simulando la aritmética de punto
# fijo bit a bit contra el modelo float:
#
#     Keras float32 (referencia) ........................ 0,9913
#     ap_fixed<8,2> global + AP_WRAP (config anterior) ... 0,5223   <-- roto
#     esta configuración ................................ 0,9933
#
# Tres causas independientes, y hay que corregir las TRES:
#
#   1. granularity='model' no emite ninguna entrada por capa, así que las
#      QActivation pierden el ufixed<4,0,RND_CONV,SAT> que hls4ml deriva de
#      sus cuantizadores quantized_relu(4,0). Abajo se usa 'name'.
#
#   2. AP_WRAP es el modo por defecto de ap_fixed: un valor fuera de rango
#      envuelve a un negativo pequeño en vez de saturar. Las salidas de
#      BatchNorm llegan a +34,7, así que activaciones que en entrenamiento
#      saturan a 0,9375 envuelven a casi cero en hardware. Con los anchos
#      correctos pero AP_WRAP la exactitud solo llega a 0,6026.
#
#   3. ap_fixed<8,2> representa [-2,+2) y los rangos reales no se le parecen:
#      salida de qdense_1 [-325,6 · +275,2], BatchNorm [-151,6 · +34,7],
#      acumuladores hasta 10 bits enteros.
#
# El valor por defecto propio de hls4ml ya es fixed<16,6>: la configuración
# anterior sustituyó un defecto sensato por uno roto.
PRECISION_DEFAULT = 'ap_fixed<16,6,AP_RND,AP_SAT>'
PRECISION_INPUT   = 'ap_fixed<16,6,AP_RND,AP_SAT>'   # 40,4% de paquetes salen de [-2,+2)
PRECISION_ACCUM   = 'ap_fixed<18,10,AP_RND,AP_SAT>'  # |max| medido: 325,7 / 88,7 / 20,1
PRECISION_RESULT  = 'ap_fixed<16,6,AP_RND,AP_SAT>'   # el sigmoide solo abarca [0,377 · 0,786]

# NO se sobrescriben a propósito las capas QActivation: hls4ml deriva su tipo
# del cuantizador QKeras, que es exactamente lo que hace falta. Sobrescribirlas
# es lo que rompió la conversión anterior.

# Factores de reúso por capa. hls4ml solo admite divisores de (n_in * n_out);
# el ReuseFactor=32 global anterior no lo es para qdense_1 ni para output, así
# que hls4ml los sustituía en silencio por 36 y 16. Se declaran explícitos.
FACTORES_REUSO = {
    'qdense_1': 36,   # 18 x 32 = 576
    'qdense_2': 32,   # 32 x 16 = 512
    'output'  : 16,   # 16 x  1 =  16
}
ESTRATEGIA = 'Resource'   # 'Latency' prioriza velocidad; 'Resource', área

print(f"--- Iniciando Pipeline HLS ---")
print(f"hls4ml        : {hls4ml.__version__}")
print(f"TensorFlow    : {tf.__version__}")
print(f"Modelo        : {RUTA_MODELO}")
print(f"Salida        : {DIR_HLS_OUT}")
print(f"Reloj         : {PERIODO_RELOJ} ns ({1000/PERIODO_RELOJ:.0f} MHz)")
print(f"Precisión base: {PRECISION_DEFAULT}")

In [4]:
# =====================================================================
# 2. CARGA DEL MODELO DESTILADO (Corregido para QKeras)
# =====================================================================
from qkeras.utils import load_qmodel

print(f"Cargando modelo cuantizado desde: {RUTA_MODELO}...")

# Keras puro fallaría aquí. Usamos load_qmodel de QKeras que ya conoce 
# toda la topología de hardware (QDense, QActivation, etc.)
modelo_keras = load_qmodel(RUTA_MODELO, compile=False)

modelo_keras.summary()

Cargando modelo cuantizado desde: /mnt/Datos/Julian_Alvarez-TFG/IDS-IOT/models/Destillation_15epochs_128batch/modelo_estudiante_destilado_qat.h5...


2026-08-22 15:33:26.797987: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-22 15:33:26.799482: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1956] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 qdense_1 (QDense)           (None, 32)                608       
                                                                 
 batch_normalization (BatchN  (None, 32)               128       
 ormalization)                                                   
                                                                 
 q_activation (QActivation)  (None, 32)                0         
                                                                 
 qdense_2 (QDense)           (None, 16)                528       
                                                                 
 batch_normalization_1 (Batc  (None, 16)               64        
 hNormalization)                                                 
                                                                 
 q_activation_1 (QActivation  (None, 16)               0

In [ ]:
# =====================================================================
# 3. CONFIGURACIÓN DEL PERFILADOR HLS4ML
# =====================================================================
# granularity='name' es la clave: hace que hls4ml lea el cuantizador de cada
# QDense/QActivation y asigne anchos por capa, en lugar de colapsar toda la
# red sobre una única precisión global.
config_hls = hls4ml.utils.config_from_keras_model(modelo_keras, granularity='name')

config_hls['Model']['Strategy'] = ESTRATEGIA


def _fijar_precision(cfg, capa, clave, valor):
    """Fija un campo de precisión tolerando los dos formatos de hls4ml.

    Según la versión y el tipo de capa, cfg['LayerName'][capa]['Precision'] es
    o bien una cadena suelta (que significa 'result') o bien un diccionario
    con claves weight/bias/result/accum.
    """
    entrada = cfg['LayerName'].setdefault(capa, {})
    prec = entrada.get('Precision')
    if isinstance(prec, dict):
        prec[clave] = valor
    elif prec is None:
        entrada['Precision'] = {clave: valor}
    else:
        entrada['Precision'] = {} if clave == 'result' else {'result': prec}
        entrada['Precision'][clave] = valor


# --- Valor por defecto del modelo -----------------------------------------
# Se fija SIN pisar las entradas por capa. hls4ml lo guarda como cadena suelta
# (0.7.x) o como {'default': ...} (1.x), así que se contemplan ambos.
if isinstance(config_hls['Model'].get('Precision'), dict):
    config_hls['Model']['Precision']['default'] = PRECISION_DEFAULT
else:
    config_hls['Model']['Precision'] = PRECISION_DEFAULT

# --- Puerto de entrada -----------------------------------------------------
_capa_in = modelo_keras.layers[0].name
_tensor_in = _capa_in + '_input' if _capa_in + '_input' in config_hls['LayerName'] else _capa_in
_fijar_precision(config_hls, _tensor_in, 'result', PRECISION_INPUT)

# --- Acumuladores y factores de reúso de las capas densas -------------------
for _capa, _reuso in FACTORES_REUSO.items():
    if _capa in config_hls['LayerName']:
        _fijar_precision(config_hls, _capa, 'accum', PRECISION_ACCUM)
        config_hls['LayerName'][_capa]['ReuseFactor'] = _reuso
    else:
        print(f"  AVISO: la capa {_capa!r} no existe en el modelo, se omite")

# --- Puntuación final (sigmoide) -------------------------------------------
for _nombre in config_hls['LayerName']:
    if 'sigmoid' in _nombre.lower():
        _fijar_precision(config_hls, _nombre, 'result', PRECISION_RESULT)

# --- Red de seguridad sobre las QActivation --------------------------------
# hls4ml 0.7.x y 1.x derivan el tipo del cuantizador QKeras de formas algo
# distintas. En lugar de confiar en ello, se comprueba y, si alguna capa se
# quedó sin tipo saturante, se reconstruye a partir del propio cuantizador.
# Sin AP_SAT la exactitud cae a ~0,60 (ver el bloque D de la celda anterior).
def _precision_desde_cuantizador(capa):
    q = getattr(capa, 'quantizer', None)
    bits = getattr(q, 'bits', None)
    entero = getattr(q, 'integer', None)
    if bits is None or entero is None:
        return None
    sin_signo = type(q).__name__ == 'quantized_relu'
    tipo = 'ap_ufixed' if sin_signo else 'ap_fixed'
    return f'{tipo}<{int(bits)},{int(entero)},AP_RND_CONV,AP_SAT>'


_reparadas = []
for _capa in modelo_keras.layers:
    if type(_capa).__name__ != 'QActivation':
        continue
    _prec = (config_hls['LayerName'].get(_capa.name) or {}).get('Precision')
    _txt = str(_prec.get('result') if isinstance(_prec, dict) else _prec)
    if 'SAT' in _txt.upper():
        continue
    _nueva = _precision_desde_cuantizador(_capa)
    if _nueva is None:
        raise RuntimeError(
            f"La capa {_capa.name!r} perdió su tipo saturante y no se pudo leer su "
            f"cuantizador para reconstruirlo. Sin AP_SAT la exactitud cae a ~0,60."
        )
    _fijar_precision(config_hls, _capa.name, 'result', _nueva)
    _reparadas.append((_capa.name, _txt, _nueva))

if _reparadas:
    print("\n--- QActivation reconstruidas desde el cuantizador QKeras ---")
    for _n, _antes, _ahora in _reparadas:
        print(f"  {_n:<20} {_antes}  ->  {_ahora}")

# --- Configuración efectiva ------------------------------------------------
print("\n--- Configuración Matemática Generada ---")
print(f"  {'capa':<24} {'precisión':<52} reuso")
print("  " + "-" * 86)
for _nombre, _entrada in config_hls['LayerName'].items():
    print(f"  {_nombre:<24} {str(_entrada.get('Precision')):<52} {_entrada.get('ReuseFactor')}")
print(f"\n  Model.Precision = {config_hls['Model'].get('Precision')}")
print(f"  Model.Strategy  = {config_hls['Model'].get('Strategy')}")

In [ ]:
# =====================================================================
# 4. CONVERSIÓN Y ESCRITURA DEL FIRMWARE
# =====================================================================
import shutil

print("\n[1/3] Preparando entorno y convirtiendo modelo Keras a C++...")

if os.path.exists(DIR_HLS_OUT):
    print(f"      -> Detectada compilación previa. Limpiando directorio: {DIR_HLS_OUT}")
    shutil.rmtree(DIR_HLS_OUT)

hls_model = hls4ml.converters.convert_from_keras_model(
    modelo_keras,
    hls_config=config_hls,
    output_dir=DIR_HLS_OUT,
    part=TARGET_PART,
    # board=TARGET_BOARD,
    backend='Vitis',
    io_type=IO_PROTOCOL,
    clock_period=PERIODO_RELOJ,   # sin esto hls4ml usa 5 ns (200 MHz)
)

print("[2/3] Escribiendo el proyecto HLS...")
hls_model.write()

print("[3/3] Compilando el modelo en C++ (biblioteca de simulación)...")
hls_model.compile()

print(f"\nFirmware escrito en: {DIR_HLS_OUT}/firmware")

# NOTA: la celda siguiente hace el post-proceso obligatorio (ruta de pesos y
# verificación de tipos). No te la saltes: compile() vuelve a copiar el árbol
# nnet_utils/ y deshace el parche si se aplicó antes.

In [ ]:
# =====================================================================
# 4b. POST-PROCESO DEL FIRMWARE  (obligatorio)
# =====================================================================
# (i)  Ruta absoluta de los pesos dentro de nnet_helpers.h
# (ii) Verificación de que los tipos emitidos son los esperados
# ---------------------------------------------------------------------

# --- (i) WEIGHTS_DIR -------------------------------------------------------
# En simulación C, hls4ml deja los arreglos de pesos sin inicializar y los
# carga en ejecución de firmware/weights/*.txt; bajo __SYNTHESIS__ los valores
# están empotrados en las cabeceras y esto se ignora. NO afecta al hardware.
#
# El macro tiene que expandirse a un literal de cadena. Pasarlo como
# -DWEIGHTS_DIR=\"...\" funciona desde Tcl, pero el diálogo "Edit CFLAGS" de la
# GUI de Vitis 2022.2 vuelve a dividir el campo en tokens y elimina las
# comillas escapadas, dejando una ruta desnuda que no compila. Reescribir el
# valor por defecto de la propia cabecera evita el problema: así la GUI no
# necesita ningún -D.
_helpers = os.path.join(DIR_HLS_OUT, 'firmware', 'nnet_utils', 'nnet_helpers.h')
_pesos   = os.path.abspath(os.path.join(DIR_HLS_OUT, 'firmware', 'weights'))

_viejo = '#define WEIGHTS_DIR "weights"'
_nuevo = f'#define WEIGHTS_DIR "{_pesos}"'

with open(_helpers) as _fh:
    _txt = _fh.read()

if _viejo in _txt:
    with open(_helpers, 'w') as _fh:
        _fh.write(_txt.replace(_viejo, _nuevo))
    print(f"WEIGHTS_DIR -> {_pesos}")
elif _nuevo in _txt:
    print(f"WEIGHTS_DIR ya apuntaba a {_pesos}")
else:
    print("AVISO: no se encontró el valor por defecto de WEIGHTS_DIR en nnet_helpers.h.")
    print("       La simulación C fallará con 'ERROR: file w2.txt does not exist'.")

# --- (ii) Verificación de los tipos emitidos -------------------------------
_defines = os.path.join(DIR_HLS_OUT, 'firmware', 'defines.h')
print(f"\n--- {_defines} ---")
_tipos = {}
with open(_defines) as _fh:
    for _linea in _fh:
        if _linea.startswith('typedef'):
            print("  " + _linea.rstrip())
            _partes = _linea.rstrip().rstrip(';').split()
            if len(_partes) >= 3:
                _tipos[_partes[-1]] = ' '.join(_partes[1:-1])

# Comprobaciones que atrapan las tres causas del bloque D.
_problemas = []

_entrada = _tipos.get('input_t', '')
if 'SAT' not in _entrada.upper():
    _problemas.append(f"input_t sin saturación: {_entrada}")

_activaciones = [(_n, _t) for _n, _t in _tipos.items() if 'ufixed' in _t]
if not _activaciones:
    _problemas.append("no se emitió ningún tipo ap_ufixed: las QActivation "
                      "perdieron su cuantizador")
for _n, _t in _activaciones:
    if 'SAT' not in _t.upper():
        _problemas.append(f"activación {_n} sin saturación: {_t}")

for _n, _t in _tipos.items():
    if 'ap_fixed<8,2' in _t.replace(' ', ''):
        _problemas.append(f"{_n} sigue en ap_fixed<8,2>: {_t}")

if _problemas:
    print("\n" + "!" * 72)
    print(" FIRMWARE SOSPECHOSO - no lo sintetices todavía")
    for _p in _problemas:
        print(f"   - {_p}")
    print(" Revisa la celda 3. Con estos tipos la exactitud esperada es ~0,52.")
    print("!" * 72)
else:
    print("\nOK: tipos saturantes en entrada y activaciones, sin residuos de <8,2>.")
    print("    Exactitud simulada esperada: ~0,9933 (referencia float32: 0,9913)")

In [ ]:
# =====================================================================
# 5. SÍNTESIS DE COMPROBACIÓN  (opcional)
# =====================================================================
# ATENCIÓN: esto sintetiza 'myproject' A SOLAS, sin el envoltorio ids_axi.
# Sirve como estimación rápida de recursos y temporización de la red, NO
# produce el IP core que se lleva a Vivado.
#
# El IP definitivo -- con interfaces AXI-Stream, AXI-Lite y el puerto led_o --
# lo construye el envoltorio de vitis/ids_axi:
#
#     cd vitis/ids_axi
#     vitis_hls -f scripts/build_ip.tcl
#
# Ver notebooks/Etapa_5/docs/ip_core_vitis_y_overlay_pynq.md, etapa A.
# ---------------------------------------------------------------------
EJECUTAR_SINTESIS = False   # ponlo en True para la estimación de recursos

if EJECUTAR_SINTESIS:
    print("[*] Iniciando síntesis C de la red (varios minutos)...")
    compilation = hls_model.build(csim=False, synth=True, export=False)

    if compilation:
        print("\n--- Síntesis HLS completada ---")
        print(f"-> {DIR_HLS_OUT}/myproject_prj/solution1/syn/report/")
        print("\nPresupuesto del xc7z020: 53.200 LUT / 106.400 FF / 220 DSP / 280 BRAM_18K")
        print("Los DSP son el recurso crítico. Si te pasas de 220, sube los")
        print("FACTORES_REUSO de la celda 1 y vuelve a ejecutar desde la celda 3.")
else:
    print("Síntesis omitida (EJECUTAR_SINTESIS = False).")
    print("El firmware ya está generado y verificado; continúa con build_ip.tcl.")

# NOTA sobre un error inocuo que verás en el log de esta celda:
#   ERROR: [HLS 200-642] The 'config_array_partition -maximum_size' command
#                        is not supported.
# Es una opción heredada de Vivado HLS que Vitis HLS 2022.2 eliminó. La emite
# el build_prj.tcl propio de hls4ml y no aborta el flujo. scripts/build_ip.tcl
# nunca la emite.

In [8]:
hls4ml.report.read_vivado_report(f'{DIR_HLS_OUT}/')

Found 1 solution(s) in hls_Prj/pynq_modelo_estudiante_destilado_qat_prj//myproject_prj.
Reports for solution "solution1":

C simulation report not found.
Synthesis report not found.
Co-simulation report not found.
